## 📘 1. Basic Agent
**Goal**: Create an agent using the **Foundry Project SDK** (`azure-ai-projects`) so it is registered server-side with full **tracing in the Foundry portal**, then connect to it from the **Microsoft Agent Framework** via `FoundryAgent`.

In [12]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from dotenv import load_dotenv
load_dotenv()

from utils.config import get_env, get_credential

# Endpoints
PROJECT_ENDPOINT = get_env("AZURE_AI_PROJECT_ENDPOINT_FULL")  # for AIProjectClient
MODEL                 = get_env("AZURE_AI_MODEL_DEPLOYMENT_NAME")

AGENT_NAME = "ContosoConcierge-af"

print(f"📡 Project endpoint : {PROJECT_ENDPOINT}")
print(f"🧠 Model:                   {MODEL}")

📡 Project endpoint : https://foundry-demo-us-east-resource.services.ai.azure.com/api/projects/prj-foundry-demo-us-east-dev-001
🧠 Model:                   gpt-4o


In [2]:
# Basic Agent
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient

client = FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL,
        credential=get_credential(),
    )

# client = OpenAIChatClient()
# client = AnthropicClient()
# client = OllamaChatClient()
# client = BedrockChatClient()
# client = AzureOpenAIChatClient()
# client = FoundryChatClient()

agent = Agent(
        client=client,
        name="HelloAgent",
        instructions="You are a friendly assistant. Keep your answers brief.",
    )

result = await agent.run("What is the capital of France?")

print(f"Agent: {result}")

c:\Dibyendu\Code\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Dibyendu\Code\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


2026-06-01 19:42:36,552 - azure.ai.agentserver - INFO - Application Insights not configured, telemetry export disabled.
Agent: The capital of France is Paris.


---
### Part A — Create Agent with Foundry Project SDK 🏗️


In [3]:
# ── A1. Create the agent in Foundry ──────────────────────────────────────────
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

INSTRUCTIONS = """\
You are **Contoso Concierge**, a friendly and knowledgeable assistant for
Contoso Corporation. You help employees with general questions about company
policies, product information, and day-to-day tasks.

Guidelines:
- Be concise but thorough.
- When you don't know something, say so honestly.
- Use a professional yet warm tone.
"""

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=get_credential(),
)

agent_version = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=INSTRUCTIONS,
    ),
)

print(f"✅ Agent created in Foundry!")
print(f"   Name:    {agent_version.name}")
print(f"   Version: {agent_version.version}")
print(f"   ID:      {agent_version.id}")
print(f"\n🔍 Traces for this agent will appear in the Foundry portal.")

✅ Agent created in Foundry!
   Name:    ContosoConcierge-af
   Version: 1
   ID:      ContosoConcierge-af:1

🔍 Traces for this agent will appear in the Foundry portal.


---
### Part B — Use with Microsoft Agent Framework 🤖

In [4]:
# ── B1. Connect to the Foundry agent via Agent Framework ─────────────────────
from agent_framework import Agent
from agent_framework.foundry import FoundryAgent

af_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=agent_version.name,
    agent_version=str(agent_version.version),
    credential=get_credential(),
)

print(f"🤖 Agent Framework connected to Foundry agent: {agent_version.name} v{agent_version.version}")
print(f"   Type: {type(af_agent).__name__}")

🤖 Agent Framework connected to Foundry agent: ContosoConcierge-af v1
   Type: FoundryAgent


In [5]:
# ── B2. Single-turn call via Agent Framework ─────────────────────────────────
result = await af_agent.run("What departments does Contoso have?")
print(f"🗨️  Agent (via Agent Framework): {result}")

🗨️  Agent (via Agent Framework): Contoso Corporation has several key departments to support its operations:

1. **Sales and Marketing**: Focuses on promoting products and services, managing client relationships, and driving revenue.

2. **Research and Development (R&D)**: Works on product innovation, development, and improvement.

3. **Human Resources (HR)**: Manages recruitment, employee relations, benefits, and training programs.

4. **Finance and Accounting**: Oversees budgeting, financial reporting, and managing company expenses.

5. **Information Technology (IT)**: Provides technical support, manages IT infrastructure, and ensures data security.

6. **Customer Service**: Handles inquiries, support requests, and maintains customer satisfaction.

7. **Operations**: Manages production processes, supply chain logistics, and quality control.

If you need specifics about any department, feel free to ask!


In [6]:
# ── B3. Streaming response ───────────────────────────────────────────────────
print("🗨️  Agent (streaming): ", end="", flush=True)
async for chunk in af_agent.run("Tell me how to make a soup.", stream=True):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print()

🗨️  Agent (streaming): Making a soup can be simple and customizable. Here’s a basic guideline:

1. **Choose Your Ingredients:**
   - **Base:** Start with broth or stock (chicken, beef, vegetable).
   - **Proteins:** Chicken, beef, tofu, or legumes like beans.
   - **Vegetables:** Carrots, celery, onions, potatoes, and greens.
   - **Seasonings:** Salt, pepper, herbs (thyme, bay leaf, parsley).

2. **Preparation Steps:**
   - **Sauté Aromatics:** Heat oil or butter in a pot and sauté onions, garlic, and other aromatics.
   - **Add Proteins:** Brown meats if using, or add legumes.
   - **Build the Broth:** Pour in your chosen broth or stock.
   - **Simmer:** Bring to a boil, reduce heat, and simmer.
   - **Vegetables:** Add vegetables and cook until tender.

3. **Finishing Touches:**
   - Taste and adjust seasonings as needed.
   - For creamy soup, blend if desired.
   - Garnish with fresh herbs or a splash of lemon juice for brightness.

Feel free to adjust based on your preferences and

---
### Part C — Sessions and Conversations

In [7]:
# ── C1. Multi-turn with AgentSession ────────────────────────────
session = af_agent.create_session()
print(f"💬 Session created: {session.session_id}")

# First turn
result1 = await af_agent.run(
    "My name is Dibyendu and I work in IT department. I like chocolates.",
    session=session,
)
print(f"\n🗨️  Agent: {result1}")

# Second turn — agent should remember the name
result2 = await af_agent.run(
    "What do I like ?",
    session=session,
)
print(f"\n🗨️  Agent: {result2}")

💬 Session created: c4c8a15c-e62a-4f94-bb22-0e8447f50629

🗨️  Agent: Hello Dibyendu! It's great to meet you. If you ever need assistance with company policies, IT-related queries, or anything else, feel free to reach out. And if there's a chocolate appreciation club at Contoso, I'll keep you posted!

🗨️  Agent: You mentioned that you like chocolates! If there's anything else you'd like to share or ask about, feel free to let me know.


In [8]:
# ── C2. Multi-turn conversation with AgentSession and Foundry Conversation ────────────────────────────

# Create a new conversation in Foundry and link it to the AgentSession
conversation = project.get_openai_client().conversations.create()
print(f"\n🗨️  Conversation: {conversation.id}")



🗨️  Conversation: conv_70ae187f3382fefe00twQqMCQ967oM2uokVXITZbPQb5tPnoVE


In [9]:
# ── C3. Multi-turn with AgentSession and Foundry Conversation ────────────────────────────
session = af_agent.create_session()
session.service_session_id = conversation.id

print(f"💬 Session created: {session.session_id}")
print(f"🔗 Bound to Foundry conversation: {session.service_session_id}")

# First turn
result1 = await af_agent.run(
    "My name is Dibyendu and I work in IT department. I like chocolates.",
    session=session,
)
print(f"\n🗨️  Agent: {result1}")

# Second turn — agent should remember the name
result2 = await af_agent.run(
    "What do I like ?",
    session=session,
)
print(f"\n🗨️  Agent: {result2}")

💬 Session created: adace98c-9464-4301-a590-6786abeadf48
🔗 Bound to Foundry conversation: conv_70ae187f3382fefe00twQqMCQ967oM2uokVXITZbPQb5tPnoVE

🗨️  Agent: Hello, Dibyendu! It's great to meet you. Chocolates are a wonderful treat. If you have any questions or need assistance related to your work in the IT department or company policies, feel free to ask.

🗨️  Agent: You mentioned that you like chocolates! If there's anything else you'd like to share or need help with, let me know.


In [10]:
# ── C4. Multi-turn with AgentSession and Existing Foundry Conversation ────────────────────────────
session = af_agent.create_session()
session.service_session_id = conversation.id
print(f"💬 Session created: {session.session_id}")
print(f"🔗 Reusing Foundry conversation: {session.service_session_id}")

# First turn
result1 = await af_agent.run(
    "What is my name ?",
    session=session,
)
print(f"\n🗨️  Agent: {result1}")

# Second turn — agent should remember the name
result2 = await af_agent.run(
    "What do I like ?",
    session=session,
)
print(f"\n🗨️  Agent: {result2}")

💬 Session created: 161dbd55-82c2-4bff-9b0b-a997dab49123
🔗 Reusing Foundry conversation: conv_70ae187f3382fefe00twQqMCQ967oM2uokVXITZbPQb5tPnoVE

🗨️  Agent: Your name is Dibyendu. If there's anything else you need assistance with, feel free to ask!

🗨️  Agent: You like chocolates! If there's anything else you'd like to discuss or inquire about, I'm here to help.


---
### Part C — Cleanup 🧹

In [11]:
# ── C1. Delete the agent version from Foundry ────────────────────────────────
# Uncomment the next line to delete the agent when you're done:
project.agents.delete(agent_name=AGENT_NAME)
print("🧹 Agent deleted from Foundry")

#print("💡 Tip: Keep the agent to reuse in other notebooks or the Foundry Playground.")
print(f"   Agent name:    {AGENT_NAME}")
print(f"   Agent version: {agent_version.version}")

🧹 Agent deleted from Foundry
   Agent name:    ContosoConcierge-af
   Agent version: 1


---
### ✅ What you demonstrated

| Capability | SDK / Feature |
|---|---|
| Register agent in Foundry | `AIProjectClient` → `agents.create_version()` |
| Server-side traces | All calls visible in Foundry portal → Tracing |
| Agent Framework bridge | `FoundryAgent(agent_name=..., agent_version=...)` |
| Multi-turn memory | `AgentSession` retains context across turns |
| Streaming | `async for chunk in agent.run(..., stream=True)` |

**Next →** [02_agent_tools_functioncall.ipynb](./02_agent_tools_functioncall.ipynb) — Local Function Tools.